<a href="https://colab.research.google.com/github/saverin0/Change-Detection-Using-Dinov3/blob/main/02_feature_extraction_dinov3_sat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SpaceNet-7 Temporal Change Detection with DINOv3
## Part 2: DINOv3 SAT-493M Feature Extraction

### What this notebook does
It turns every **monthly satellite image** into a grid of **DINOv3 patch features**, the input for the change-detection model in Part 3. Each location produces one file:

| File | Array | Shape (default settings) | Meaning |
|---|---|---|---|
| `<FEATURES_SUBDIR>/<loc_id>_features.npz` | `features` | `(T, 64, 64, 256)` float16 | One feature vector per 16×16 px patch, for each of the T months |
| | `months` | `(T,)` | `YYYY_MM` for each time step, **in the same order as the Part 1 labels** |
| `<FEATURES_SUBDIR>/pca.npz` | `mean`, `components`, ... | | The PCA used to compress 1024 → 256 dims (shared by all locations) |

### The model
**`facebook/dinov3-vitl16-pretrain-sat493m`**: a ViT-Large (~300M parameters) from Meta's DINOv3, pretrained self-supervised on **493 million satellite images**. Its features already capture roofs, roads, and vegetation, so Part 3 only has to learn *what changed*.

### Resolution: why 1024 px and PCA
Each patch token covers 16×16 pixels of the **resized** image. A SpaceNet-7 tile is ~1024 px, about 4.8 km across:

| `FEATURE_IMAGE_SIZE` | Patch grid | Ground size of one patch | Features per location (fp16) |
|---|---|---|---|
| 512 (first run) | 32×32 | ~150 m | 1024 dims: ~50 MB |
| **1024 (default)** | **64×64** | **~75 m** | 1024 dims: ~210 MB → **PCA to 256 dims: ~50 MB** |

A single house is only ~10–20 m wide, so it barely changes a 150 m patch's features. The first run missed most scattered single-building changes, so this notebook now extracts at **full resolution (1024 px)**.

At 64×64, full 1024-dim features would take ~12.6 GB, more than Colab's RAM in Part 3. **PCA** (principal component analysis) compresses each 1024-dim vector to its **256 most informative directions**, which keeps the total at ~3 GB. Part 3's first layer projects features down to 256 dims anyway. PCA is fitted **only on training locations** from Part 1's split, so no validation or test data influences it. The cell prints how much of the variance the 256 dims keep.

To reproduce the first (32×32) run, set `FEATURE_IMAGE_SIZE = 512`, `PCA_DIM = None`, and `FEATURES_SUBDIR = "features"`.

### Pipeline at a glance
```
Mount Drive ─► Install deps ─► Load config & settings ─► Check GPU ─► Hugging Face login ─► Load DINOv3
     ─► Define helpers ─► Find locations still to do ─► Sanity check on one batch
     ─► Stage images to local SSD ─► Fit (or load) PCA ─► Extract features + upload to Drive ─► Verify
```

### Before you run
1. **Run Part 1 first**, including Step 14 (the split).
2. **Use a GPU runtime:** Runtime → Change runtime type → **T4 GPU**.
3. **Get model access:** accept the license on the [Hugging Face model page](https://huggingface.co/facebook/dinov3-vitl16-pretrain-sat493m), create a read token, and add it to **Colab Secrets** (🔑) as `HF_TOKEN`, with notebook access switched on.
4. **Drive space:** ~3 GB for the new features folder. The old `features/` folder isn't touched.

## Step 1: Mount Google Drive

The raw images and Part 1's outputs are on Drive, and the features are written back there so Part 3 can use them after this runtime is gone.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


## Step 2: Install dependencies and import libraries

Colab already has PyTorch, NumPy, and tqdm. Two things may need installing:
- **`transformers` ≥ 4.56**, the first version with DINOv3 support. It's upgraded only if the installed version is older.
- **`rasterio`**, which reads the GeoTIFF images.

> If this cell upgrades `transformers` after it has already been imported in this session, restart the runtime (Runtime → Restart session) and run all cells again.

In [2]:
import importlib.metadata
import importlib.util
import subprocess
import sys


def installed_version(package):
    try:
        return tuple(int(part) for part in importlib.metadata.version(package).split(".")[:2])
    except importlib.metadata.PackageNotFoundError:
        return None


to_install = []
transformers_version = installed_version("transformers")
if transformers_version is None or transformers_version < (4, 56):
    to_install.append("transformers>=4.56")
if importlib.util.find_spec("rasterio") is None:
    to_install.append("rasterio")
if to_install:
    print(f"Installing: {', '.join(to_install)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *to_install])

import json
import shutil
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import rasterio
import torch
from rasterio.enums import Resampling
from tqdm.auto import tqdm

print(f"transformers {importlib.metadata.version('transformers')} | torch {torch.__version__}")

transformers 5.16.1 | torch 2.11.0+cu128


## Step 3: Load the Part 1 configuration and choose the feature settings

Shared paths come from Part 1's `config.json`. The feature settings are chosen here:

| Setting | Default | Meaning |
|---|---|---|
| `FEATURE_IMAGE_SIZE` | 1024 | Images are resized to this before DINOv3. It must be a multiple of the patch size (16). **1024 → 64×64 patches** |
| `PCA_DIM` | 256 | Dims kept per patch after PCA. `None` saves the raw 1024 dims |
| `FEATURES_SUBDIR` | `features_1024_pca256` | Output folder inside the cache. Part 3 points to it. Use a **new name** whenever you change the settings above |
| `PCA_SAMPLE_EVERY_NTH_MONTH` | 3 | PCA is fitted on every 3rd month of every training location (~340 images, ~1.4 million patch vectors) |

`CACHE_BASE` must match Part 1.

In [3]:
MODEL_NAME = "facebook/dinov3-vitl16-pretrain-sat493m"

# Must match cache_base from Part 1.
CACHE_BASE = Path("/content/drive/MyDrive/datasets/spacenet7_cache")

# First (32x32) run: FEATURE_IMAGE_SIZE = 512, PCA_DIM = None, FEATURES_SUBDIR = "features"
FEATURE_IMAGE_SIZE = 1024
PCA_DIM = 256
FEATURES_SUBDIR = "features_1024_pca256"
PCA_SAMPLE_EVERY_NTH_MONTH = 3

config_path = CACHE_BASE / "config.json"
if not config_path.exists():
    raise FileNotFoundError(
        f"{config_path} not found. Run 01_config_and_labels.ipynb first, with the same cache_base."
    )
config = json.loads(config_path.read_text())

if config["dino_model"] != MODEL_NAME:
    raise ValueError(f"config.json expects {config['dino_model']}, but this notebook uses {MODEL_NAME}.")

DATA_ROOT = Path(config["data_root"])
CACHE_DIR = Path(config["cache_dir"])
LABELS_DIR = CACHE_DIR / "labels"
SPLIT_PATH = CACHE_DIR / "training" / "split.json"
FEATURES_DIR = CACHE_DIR / FEATURES_SUBDIR
PCA_PATH = FEATURES_DIR / "pca.npz"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

PATCH_SIZE = config["patch_size"]
if FEATURE_IMAGE_SIZE % PATCH_SIZE != 0:
    raise ValueError(f"FEATURE_IMAGE_SIZE {FEATURE_IMAGE_SIZE} must be a multiple of patch_size {PATCH_SIZE}.")
PATCHES_PER_SIDE = FEATURE_IMAGE_SIZE // PATCH_SIZE
N_PATCHES = PATCHES_PER_SIDE ** 2

LOCAL_IMAGES = Path("/content/sn7_part2_images")
LOCAL_FEATURES = Path("/content/sn7_part2_features")
IMAGE_LOAD_THREADS = 4

print("Configuration:")
print(f"   Model: {MODEL_NAME}")
print(f"   Data root: {DATA_ROOT}")
print(f"   Labels: {LABELS_DIR}")
print(f"   Features: {FEATURES_DIR}")
print(f"   Image size: {FEATURE_IMAGE_SIZE} -> {PATCHES_PER_SIDE}x{PATCHES_PER_SIDE} patches")
print(f"   PCA: {'off (raw features)' if PCA_DIM is None else f'{PCA_DIM} dims'}")

Configuration:
   Model: facebook/dinov3-vitl16-pretrain-sat493m
   Data root: /content/drive/MyDrive/datasets/spacenet7/SN7_buildings_train/train
   Labels: /content/drive/MyDrive/datasets/spacenet7_cache/labels
   Features: /content/drive/MyDrive/datasets/spacenet7_cache/features_1024_pca256
   Image size: 1024 -> 64x64 patches
   PCA: 256 dims


## Step 4: Check the GPU and pick a batch size

On a CPU, extraction would take many hours, so this cell stops with an error if no GPU is attached.

**Batch size** is how many images go through the model at once. It depends on the token count:

| Image size | Tokens per image | Batch (≥ 14 GB GPU, e.g. T4) | Batch (smaller GPU) |
|---|---|---|---|
| 512 | 1,029 | 16 | 8 |
| 1024 | 4,101 | 8 | 4 |

At 1024 px each image has 4× the tokens, and attention cost grows faster than that, so the batch is halved. If you get `CUDA out of memory`, lower `BATCH_SIZE` and re-run from this step.

In [4]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. In Colab: Runtime -> Change runtime type -> T4 GPU, then run all cells again."
    )

device = torch.device("cuda")
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
large_gpu = gpu_memory_gb >= 14
if N_PATCHES <= 1024:
    BATCH_SIZE = 16 if large_gpu else 8
else:
    BATCH_SIZE = 8 if large_gpu else 4

print(f"GPU: {torch.cuda.get_device_name()} ({gpu_memory_gb:.1f} GB)")
print(f"Batch size: {BATCH_SIZE}")

GPU: Tesla T4 (15.6 GB)
Batch size: 8


## Step 5: Log in to Hugging Face

DINOv3 weights are gated, so downloading them needs your Hugging Face token. It's read from **Colab Secrets** (`HF_TOKEN`), so it never appears in the notebook or its saved outputs.

> **Never paste the token into a cell.** When you save this notebook to GitHub from Colab, cell contents and outputs are saved with it.

In [5]:
from google.colab import userdata
from huggingface_hub import login

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception as error:
    raise RuntimeError(
        "Could not read the HF_TOKEN secret. Add it under the key icon in the left sidebar "
        "and switch on notebook access."
    ) from error

login(token=hf_token, add_to_git_credential=False)
del hf_token
print("Logged in to Hugging Face.")

Logged in to Hugging Face.


## Step 6: Load DINOv3 SAT-493M

This downloads the model (~1.2 GB, cached for the rest of the session) and puts it on the GPU in eval mode.

**Precision:** the weights stay in float32, and the forward pass runs under **fp16 autocast**. The T4 has fast fp16 math, and autocast keeps numerically sensitive ops like LayerNorm and softmax in float32, avoiding fp16 overflow. The T4 doesn't support bfloat16.

**Attention:** the model is loaded with `attn_implementation="sdpa"` (PyTorch's scaled-dot-product attention). At 1024 px every image has 4,101 tokens, and plain ("eager") attention would materialize a 4,101×4,101 matrix per head, which is over 4 GB per batch. SDPA's memory-efficient kernel never builds that matrix and runs on the T4. The cell prints which implementation is active.

**Preprocessing:** the Hugging Face image processor resizes to **224×224** by default, which would give tiny patch grids. So only its **normalization values** (`image_mean` / `image_std`, the satellite-specific values for this model) are taken from it. Resizing to `FEATURE_IMAGE_SIZE` and normalization are done by our own code in Step 7.

In [6]:
from transformers import AutoImageProcessor, AutoModel

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
try:
    model = AutoModel.from_pretrained(MODEL_NAME, attn_implementation="sdpa")
except (ValueError, TypeError):
    model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device).eval()
ATTENTION = getattr(model.config, "_attn_implementation", "unknown")

HIDDEN_DIM = model.config.hidden_size
N_REGISTER_TOKENS = getattr(model.config, "num_register_tokens", 0)

if model.config.patch_size != PATCH_SIZE:
    raise ValueError(f"Model patch size {model.config.patch_size} != config patch_size {PATCH_SIZE}.")
if HIDDEN_DIM != config["hidden_dim"]:
    raise ValueError(f"Model hidden size {HIDDEN_DIM} != config hidden_dim {config['hidden_dim']}.")
if PCA_DIM is not None and not 0 < PCA_DIM < HIDDEN_DIM:
    raise ValueError(f"PCA_DIM must be between 1 and {HIDDEN_DIM - 1}, or None.")

PIXEL_MEAN = torch.tensor(processor.image_mean, device=device).view(1, 3, 1, 1)
PIXEL_STD = torch.tensor(processor.image_std, device=device).view(1, 3, 1, 1)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Loaded {MODEL_NAME} ({n_params:.0f}M parameters)")
print(f"   Hidden dim: {HIDDEN_DIM} | Patch size: {PATCH_SIZE} | Register tokens: {N_REGISTER_TOKENS}")
print(f"   Tokens per image: 1 CLS + {N_REGISTER_TOKENS} registers + {N_PATCHES} patches")
print(f"   Attention: {ATTENTION}")
print(f"   Normalization mean={processor.image_mean} std={processor.image_std}")
if ATTENTION == "eager" and N_PATCHES > 1024:
    print("   WARNING: eager attention materializes the full attention matrix. At this image size it may "
          "run out of GPU memory; upgrade transformers or lower BATCH_SIZE.")

preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

Loaded facebook/dinov3-vitl16-pretrain-sat493m (303M parameters)
   Hidden dim: 1024 | Patch size: 16 | Register tokens: 4
   Tokens per image: 1 CLS + 4 registers + 4096 patches
   Attention: sdpa
   Normalization mean=(0.43, 0.411, 0.296) std=(0.213, 0.156, 0.143)


## Step 7: Define the helper functions

| Function | What it does |
|---|---|
| `month_of` | Gets `YYYY_MM` from an image filename (`global_monthly_2018_01_...tif`) |
| `image_dir` | Picks `images_masked/` (cloud-masked, preferred) or falls back to `images/` |
| `load_image` | Reads a GeoTIFF with rasterio, resizing to `FEATURE_IMAGE_SIZE` **during the read** (bilinear). Keeps the RGB bands and drops alpha. Returns `uint8` `(3, H, W)` |
| `patch_tokens` | Moves a batch to the GPU, scales to 0–1, normalizes, runs DINOv3 under fp16 autocast, and returns only the patch tokens `(B, N_PATCHES, 1024)` in float32 |
| `to_features` | Applies PCA if it's enabled, then reshapes to `(B, 64, 64, dims)` float16 on the CPU |
| `extract_location` | Loads all of a location's months with a thread pool and runs them through `patch_tokens` → `to_features` in batches |

`patch_tokens` checks that the model returned exactly `1 + registers + N_PATCHES` tokens, so a wrong image size fails loudly.

The PCA parameters (`pca_mean`, `pca_components`) start as `None` and are set in Step 11. They're applied on the GPU as `(x − mean) · componentsᵀ`.

In [7]:
def month_of(filename):
    parts = Path(filename).stem.split("_")
    if len(parts) >= 4 and parts[0] == "global" and parts[1] == "monthly":
        return f"{parts[2]}_{parts[3]}"
    return None


def image_dir(location):
    masked = location / "images_masked"
    return masked if masked.exists() else location / "images"


def load_image(path, size=FEATURE_IMAGE_SIZE):
    with rasterio.open(path) as src:
        img = src.read(out_shape=(src.count, size, size), resampling=Resampling.bilinear)
    img = img[:3] if img.shape[0] >= 3 else np.repeat(img[:1], 3, axis=0)
    if img.dtype != np.uint8:
        img = np.nan_to_num(img.astype(np.float32))
        lo, hi = img.min(), img.max()
        img = (img - lo) / (hi - lo) * 255 if hi > lo else np.zeros_like(img)
        img = img.astype(np.uint8)
    return img


@torch.inference_mode()
def patch_tokens(images):
    pixels = torch.from_numpy(images).to(device).float().div_(255)
    pixels = (pixels - PIXEL_MEAN) / PIXEL_STD
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        tokens = model(pixel_values=pixels).last_hidden_state

    expected_tokens = 1 + N_REGISTER_TOKENS + N_PATCHES
    if tokens.shape[1] != expected_tokens:
        raise RuntimeError(f"Model returned {tokens.shape[1]} tokens, expected {expected_tokens}.")
    return tokens[:, 1 + N_REGISTER_TOKENS:, :].float()


pca_mean = None
pca_components = None


@torch.inference_mode()
def to_features(patches):
    if pca_components is not None:
        patches = (patches - pca_mean) @ pca_components.T
    patches = patches.reshape(len(patches), PATCHES_PER_SIDE, PATCHES_PER_SIDE, -1)
    return patches.to(torch.float16).cpu().numpy()


def extract_location(image_paths, load_pool):
    images = list(load_pool.map(load_image, image_paths))
    batches = [
        to_features(patch_tokens(np.stack(images[i:i + BATCH_SIZE])))
        for i in range(0, len(images), BATCH_SIZE)
    ]
    return np.concatenate(batches)

## Step 8: Find the locations that still need features

The work list comes from **Part 1's label files**. Each label file stores the exact list of `months` it covers, and features are extracted for those months in that order. That keeps `features[t]` lined up with `building_masks[t]` in Part 3.

Locations whose `<loc_id>_features.npz` is already in `FEATURES_SUBDIR` are skipped. If Colab disconnects, run all cells again and it resumes.

A safety check: if PCA is on and some feature files exist but `pca.npz` doesn't, the folder is inconsistent (those files were made with a PCA that no longer exists), so the cell stops.

In [8]:
label_files = sorted(LABELS_DIR.glob("*_labels.npz"))
if not label_files:
    raise FileNotFoundError(f"No label files in {LABELS_DIR}. Run 01_config_and_labels.ipynb first.")

existing_features = sorted(FEATURES_DIR.glob("*_features.npz"))
if PCA_DIM is not None and existing_features and not PCA_PATH.exists():
    raise RuntimeError(
        f"{FEATURES_DIR} has {len(existing_features)} feature files but no pca.npz. "
        "Use a new FEATURES_SUBDIR or delete the old files."
    )

jobs = []
already_done = 0
for label_file in label_files:
    loc_id = label_file.name.removesuffix("_labels.npz")
    if (FEATURES_DIR / f"{loc_id}_features.npz").exists():
        already_done += 1
        continue
    with np.load(label_file, allow_pickle=True) as labels:
        months = [str(month) for month in labels["months"]]
    jobs.append({"loc_id": loc_id, "months": months})

n_images = sum(len(job["months"]) for job in jobs)
print(f"Locations with labels: {len(label_files)}")
print(f"   Already done: {already_done}")
print(f"   To process: {len(jobs)} ({n_images} images)")

Locations with labels: 60
   Already done: 0
   To process: 60 (1423 images)


## Step 9: Sanity check on one batch

Before the full run, this pushes a few images through the model and checks:
- the output has `N_PATCHES` patch tokens of 1024 dims (4,096 at 1024 px)
- there are **no NaN/Inf values**, which would point to an fp16 overflow
- how long each image takes, giving a **rough time estimate**

The first call is a warm-up (CUDA kernel selection), so only the second call is timed. The estimate covers GPU and image-loading time only. Copying from Drive (Step 10), fitting PCA (Step 11, ~340 extra images), and uploads add to it.

In [9]:
sample_loc = jobs[0]["loc_id"] if jobs else label_files[0].name.removesuffix("_labels.npz")
sample_paths = sorted(image_dir(DATA_ROOT / sample_loc).glob("*.tif"))[:4]
sample_images = np.stack([load_image(path) for path in sample_paths])

patch_tokens(sample_images)
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
sample_patches = patch_tokens(sample_images)
torch.cuda.synchronize()
seconds_per_image = (time.perf_counter() - start) / len(sample_images)

expected_shape = (len(sample_images), N_PATCHES, HIDDEN_DIM)
if tuple(sample_patches.shape) != expected_shape:
    raise RuntimeError(f"Patch tokens {tuple(sample_patches.shape)} != expected {expected_shape}.")
if not torch.isfinite(sample_patches).all():
    raise RuntimeError("Patch tokens contain NaN/Inf values.")

print(f"Sample location: {sample_loc}")
print(f"   Input batch: {sample_images.shape} {sample_images.dtype}")
print(f"   Patch tokens: {tuple(sample_patches.shape)}")
print(f"   Value range: {sample_patches.min().item():.2f} to {sample_patches.max().item():.2f}")
print(f"   Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
print(f"   ~{seconds_per_image:.2f} s/image -> about {n_images * seconds_per_image / 60:.0f} min of GPU time for {n_images} images")
del sample_patches

Sample location: L15-0331E-1257N_1327_3160_13
   Input batch: (4, 3, 1024, 1024) uint8
   Patch tokens: (4, 4096, 1024)
   Value range: -1.99 to 1.95
   Peak GPU memory: 1.85 GB
   ~0.31 s/image -> about 7 min of GPU time for 1423 images


## Step 10: Stage the images from Drive onto Colab's local disk

Reading GeoTIFFs straight from Drive during extraction would leave the GPU waiting on the network. So the needed images are first copied to Colab's local SSD with **32 threads at once**; the copy is mostly network waiting, so many threads help even on 2 CPU cores.

- **Only the months in each location's label file are copied**, only for locations not yet done. For all 60 locations that's ~1,400 images, about **6 GB**.
- A location with a label month that has **no matching image** is reported and skipped. It won't be processed with misaligned months.
- Files already on local disk with the same size aren't copied again.

In [10]:
def copy_if_stale(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists() or target.stat().st_size != source.stat().st_size:
        shutil.copy2(source, target)


copy_jobs = []
ready_jobs = []
errors = []

for job in jobs:
    source_images = {month_of(path.name): path for path in image_dir(DATA_ROOT / job["loc_id"]).glob("*.tif")}
    missing = [month for month in job["months"] if month not in source_images]
    if missing:
        errors.append((job["loc_id"], f"no image for label months {missing}"))
        continue

    job["images"] = []
    for month in job["months"]:
        source = source_images[month]
        target = LOCAL_IMAGES / job["loc_id"] / source.name
        copy_jobs.append((source, target))
        job["images"].append(target)
    ready_jobs.append(job)

with ThreadPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(copy_if_stale, source, target) for source, target in copy_jobs]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Staging images from Drive"):
        future.result()

staged_gb = sum(target.stat().st_size for _, target in copy_jobs) / 1e9
print(f"Staged {len(copy_jobs)} images ({staged_gb:.1f} GB) for {len(ready_jobs)} locations")
for loc_id, message in errors:
    print(f"   Skipped {loc_id}: {message}")

Staging images from Drive:   0%|          | 0/1423 [00:00<?, ?it/s]

Staged 1423 images (5.8 GB) for 60 locations


## Step 11: Fit (or load) the PCA

PCA finds the directions along which DINOv3's 1024-dim patch vectors vary most, and keeps the top `PCA_DIM`. Every location's features are projected with the **same** PCA, so they stay comparable.

- **If `pca.npz` already exists** in `FEATURES_SUBDIR`, it's loaded. This is what happens when you resume, and it keeps later locations consistent with earlier ones. The cell checks that it was made with the same model, image size, and `PCA_DIM`.
- **Otherwise, it's fitted:**
  1. Take every `PCA_SAMPLE_EVERY_NTH_MONTH`-th month of every **training** location in `training/split.json`. Validation and test locations are never used.
  2. Run DINOv3 on those images and accumulate the sum and outer products of all patch vectors, about 1.4 million vectors at 1024 px. Each batch's product is computed in fp32 (the T4 runs fp64 at 1/32 speed) and accumulated in fp64. That's an **exact** covariance over the whole sample, with no subsampling of patches.
  3. Eigendecompose the 1024×1024 covariance and keep the top `PCA_DIM` eigenvectors.
  4. Save `mean`, `components`, `explained_variance_ratio`, and the settings to `pca.npz`.

**Kept variance** is printed at the end. As a rule of thumb, ≥ 90% means little information is lost. If it's much lower, consider a larger `PCA_DIM`, which needs a new `FEATURES_SUBDIR`.

With `PCA_DIM = None`, this step does nothing.

In [11]:
if PCA_DIM is None:
    print(f"PCA disabled: saving raw {HIDDEN_DIM}-dim features.")
elif PCA_PATH.exists():
    with np.load(PCA_PATH) as saved:
        if (str(saved["model"]) != MODEL_NAME or int(saved["image_size"]) != FEATURE_IMAGE_SIZE
                or saved["components"].shape != (PCA_DIM, HIDDEN_DIM)):
            raise ValueError(f"{PCA_PATH} was made with different settings. Use a new FEATURES_SUBDIR.")
        pca_mean = torch.from_numpy(saved["mean"]).to(device)
        pca_components = torch.from_numpy(saved["components"]).to(device)
        kept_variance = float(saved["explained_variance_ratio"].sum())
    print(f"Loaded PCA from {PCA_PATH}: {PCA_DIM} dims keep {100 * kept_variance:.1f}% of the variance")
else:
    if not SPLIT_PATH.exists():
        raise FileNotFoundError(f"{SPLIT_PATH} not found. Run Part 1 (Step 14) to create the split.")
    train_ids = set(json.loads(SPLIT_PATH.read_text())["train"])
    fit_jobs = [job for job in ready_jobs if job["loc_id"] in train_ids]
    sample_paths = [path for job in fit_jobs for path in job["images"][::PCA_SAMPLE_EVERY_NTH_MONTH]]
    if not sample_paths:
        raise RuntimeError("No staged training-location images to fit PCA on.")

    total = torch.zeros(HIDDEN_DIM, dtype=torch.float64, device=device)
    cross = torch.zeros(HIDDEN_DIM, HIDDEN_DIM, dtype=torch.float64, device=device)
    n_vectors = 0
    with ThreadPoolExecutor(IMAGE_LOAD_THREADS) as load_pool:
        for i in tqdm(range(0, len(sample_paths), BATCH_SIZE), desc=f"Fitting PCA on {len(sample_paths)} images"):
            images = np.stack(list(load_pool.map(load_image, sample_paths[i:i + BATCH_SIZE])))
            vectors = patch_tokens(images).reshape(-1, HIDDEN_DIM)
            total += vectors.sum(0, dtype=torch.float64)
            cross += (vectors.T @ vectors).double()  # fp32 matmul (fast on T4), fp64 accumulation
            n_vectors += vectors.shape[0]

    mean = total / n_vectors
    covariance = cross / n_vectors - torch.outer(mean, mean)
    eigenvalues, eigenvectors = torch.linalg.eigh(covariance)
    order = torch.argsort(eigenvalues, descending=True)
    eigenvalues, eigenvectors = eigenvalues[order].clamp(min=0), eigenvectors[:, order]
    explained = (eigenvalues / eigenvalues.sum())[:PCA_DIM]

    pca_mean = mean.float()
    pca_components = eigenvectors[:, :PCA_DIM].T.contiguous().float()
    kept_variance = float(explained.sum())
    np.savez(PCA_PATH, mean=pca_mean.cpu().numpy(), components=pca_components.cpu().numpy(),
             explained_variance_ratio=explained.cpu().numpy(), image_size=FEATURE_IMAGE_SIZE, model=MODEL_NAME,
             n_vectors=n_vectors, fit_locations=np.array(sorted(job["loc_id"] for job in fit_jobs)))
    print(f"Fitted PCA on {n_vectors:,} patch vectors from {len(sample_paths)} images of {len(fit_jobs)} training locations")
    print(f"   {PCA_DIM} dims keep {100 * kept_variance:.1f}% of the variance "
          f"(first 10 dims: {100 * float(explained[:10].sum()):.1f}%)")
    print(f"   Saved to {PCA_PATH}")

Fitting PCA on 349 images:   0%|          | 0/44 [00:00<?, ?it/s]

Fitted PCA on 1,429,504 patch vectors from 349 images of 42 training locations
   256 dims keep 91.4% of the variance (first 10 dims: 45.6%)
   Saved to /content/drive/MyDrive/datasets/spacenet7_cache/features_1024_pca256/pca.npz


## Step 12: Extract features for every location

The main step. For each location:
1. **Load** all its months from local disk with 4 threads.
2. **Run DINOv3** batch by batch, then **apply PCA** on the GPU.
3. **Save** `features` (`(T, 64, 64, 256)` float16 by default), `months`, and `loc_id` to a local `.npz`, **uncompressed**: float features barely compress, and compression would slow every save and every Part 3 load.
4. **Upload** the file to Drive in a background thread, so the GPU starts the next location right away.
5. **Delete** the location's staged images to free disk space.

**Crash-safe uploads:** each file is copied to Drive as `<name>.partial` and renamed only when the copy is complete, so an interrupted upload never counts as done.

One location failing doesn't stop the run. It's recorded and listed at the end.

**Output size:** ~50 MB per location, about **3 GB** on Drive for all 60.

In [12]:
LOCAL_FEATURES.mkdir(parents=True, exist_ok=True)


def upload_to_drive(local_file):
    partial = FEATURES_DIR / f"{local_file.name}.partial"
    shutil.copyfile(local_file, partial)
    partial.replace(FEATURES_DIR / local_file.name)
    local_file.unlink()


uploads = {}
start = time.perf_counter()

with ThreadPoolExecutor(IMAGE_LOAD_THREADS) as load_pool, ThreadPoolExecutor(1) as upload_pool:
    for job in tqdm(ready_jobs, desc="Extracting features"):
        loc_id = job["loc_id"]
        try:
            features = extract_location(job["images"], load_pool)
            local_file = LOCAL_FEATURES / f"{loc_id}_features.npz"
            np.savez(local_file, features=features, months=np.array(job["months"]), loc_id=loc_id)
            uploads[upload_pool.submit(upload_to_drive, local_file)] = loc_id
        except Exception as error:
            errors.append((loc_id, f"{type(error).__name__}: {error}"))
        finally:
            shutil.rmtree(LOCAL_IMAGES / loc_id, ignore_errors=True)

    for future in tqdm(as_completed(uploads), total=len(uploads), desc="Uploading to Drive"):
        try:
            future.result()
        except Exception as error:
            errors.append((uploads[future], f"upload failed: {error}"))

failed = {loc_id for loc_id, _ in errors}
succeeded = [loc_id for loc_id in uploads.values() if loc_id not in failed]
elapsed_min = (time.perf_counter() - start) / 60
print(f"Done in {elapsed_min:.1f} min | Extracted: {len(succeeded)} | Already done: {already_done} | Errors: {len(errors)}")
for loc_id, message in errors:
    print(f"   {loc_id}: {message}")

Extracting features:   0%|          | 0/60 [00:00<?, ?it/s]

Uploading to Drive:   0%|          | 0/60 [00:00<?, ?it/s]

Done in 9.8 min | Extracted: 60 | Already done: 0 | Errors: 0


## Step 13: Verify the feature cache

A final check that Part 3 has everything it needs:
- **Every location with labels has a feature file.** Any missing ones are listed. Run all cells again to fill them in.
- **Months line up:** each feature file's `months` must exactly match its label file's `months`, checked for every location. Only the small `months` arrays are read.
- **Shape and dtype** of one sample file, the PCA's kept variance, and total size on Drive.

In [13]:
feature_files = sorted(FEATURES_DIR.glob("*_features.npz"))
feature_ids = {path.name.removesuffix("_features.npz") for path in feature_files}
label_ids = {path.name.removesuffix("_labels.npz") for path in label_files}
missing_ids = sorted(label_ids - feature_ids)


def months_match(loc_id):
    with np.load(LABELS_DIR / f"{loc_id}_labels.npz", allow_pickle=True) as labels, \
         np.load(FEATURES_DIR / f"{loc_id}_features.npz") as features:
        return [str(m) for m in labels["months"]] == [str(m) for m in features["months"]]


with ThreadPoolExecutor(max_workers=16) as pool:
    mismatched = [loc_id for loc_id, ok in zip(sorted(feature_ids), pool.map(months_match, sorted(feature_ids))) if not ok]

print(f"Feature files: {len(feature_files)} / {len(label_ids)} locations in {FEATURES_DIR}")
print(f"   Missing: {missing_ids if missing_ids else 'none'}")
print(f"   Month mismatches: {mismatched if mismatched else 'none'}")

if feature_files:
    with np.load(feature_files[0]) as sample:
        print(f"Sample: {feature_files[0].name}")
        print(f"   features: {sample['features'].shape} {sample['features'].dtype}")
        print(f"   months: {len(sample['months'])} ({sample['months'][0]} to {sample['months'][-1]})")
    if PCA_PATH.exists():
        with np.load(PCA_PATH) as saved:
            print(f"   PCA: {saved['components'].shape[0]} dims, {100 * float(saved['explained_variance_ratio'].sum()):.1f}% variance kept")
    total_gb = sum(path.stat().st_size for path in feature_files) / 1e9
    print(f"   Total size on Drive: {total_gb:.2f} GB")

if not missing_ids and not mismatched:
    print("\nPart 2 complete!")

Feature files: 60 / 60 locations in /content/drive/MyDrive/datasets/spacenet7_cache/features_1024_pca256
   Missing: none
   Month mismatches: none
Sample: L15-0331E-1257N_1327_3160_13_features.npz
   features: (25, 64, 64, 256) float16
   months: 25 (2018_01 to 2020_01)
   PCA: 256 dims, 91.4% variance kept
   Total size on Drive: 2.98 GB

Part 2 complete!
